# 📄 English Toxicity Detection System

Simplified NLP system for detecting toxic comments using pre-trained Detoxify model.

## Workflow\
    "Input Text (any language)\n",
    "        â†“\n",
    "Language Detection\n",
    "        â†“\n",
    "Translation to English (if needed)\n",
```
    ↓
Preprocessing (URLs, emails, whitespace)
    ↓
Detoxify Model (Pre-trained)
    ↓
6 Toxicity Labels (Probability scores)
    ↓
Severity Calculation (5 levels)
    ↓
JSON Output
```

## Quick Start
- Run cells 1-6 sequentially
- Use `pipeline.process()` for single predictions
- Use `pipeline.process_batch()` for batch processing

# 🚀 PHASE 1: Environment Setup & Dependencies

In [2]:
# Install required libraries
import subprocess
import sys

libraries = [
    'detoxify',
    'pandas',
    'numpy',
    'torch',
    'transformers',
    'scikit-learn',
    'tqdm',
    'matplotlib',
    'seaborn',
    'requests'
]

print("Installing required libraries...")
for lib in libraries:
    print(f"Installing {lib}...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", lib])
    except Exception as e:
        print(f"  ⚠️  {lib} installation issue: {str(e)[:50]}")

print("\n✅ Core libraries installed!")

Installing required libraries...
Installing detoxify...
Installing pandas...
Installing numpy...
Installing torch...
Installing transformers...
Installing scikit-learn...
Installing tqdm...
Installing matplotlib...
Installing seaborn...
Installing requests...

✅ Core libraries installed!


In [3]:
# Import all required libraries
import os
import json
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Tuple, Any
from datetime import datetime

# Detoxify for toxicity detection
from detoxify import Detoxify

# Deep Learning Libraries
import torch

# Utilities
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# Suppress warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print("✅ All imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")
print(f"Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")

C:\Users\kumar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ All imports successful!
PyTorch version: 2.1.0+cpu
GPU Available: False
Device: CPU


In [4]:
# Create project directory structure
PROJECT_ROOT = Path("d:/NLP")
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
LOGS_DIR = PROJECT_ROOT / "logs"
RESULTS_DIR = PROJECT_ROOT / "results"

# Create all directories
for directory in [DATA_DIR, RAW_DATA_DIR, PROCESSED_DATA_DIR, MODELS_DIR, LOGS_DIR, RESULTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
    print(f"✅ Created: {directory}")

# Configuration
CONFIG = {
    'project_root': str(PROJECT_ROOT),
    'data_dir': str(DATA_DIR),
    'raw_data_dir': str(RAW_DATA_DIR),
    'processed_data_dir': str(PROCESSED_DATA_DIR),
    'models_dir': str(MODELS_DIR),
    'logs_dir': str(LOGS_DIR),
    'results_dir': str(RESULTS_DIR),
    'model_name': 'detoxify-multilingual',  # Pre-trained detoxify model
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'language': 'English (Only)',
    'toxicity_labels': ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_attack']
}

print("\n📋 Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

✅ Created: d:\NLP\data
✅ Created: d:\NLP\data\raw
✅ Created: d:\NLP\data\processed
✅ Created: d:\NLP\models
✅ Created: d:\NLP\logs
✅ Created: d:\NLP\results

📋 Configuration:
  project_root: d:\NLP
  data_dir: d:\NLP\data
  raw_data_dir: d:\NLP\data\raw
  processed_data_dir: d:\NLP\data\processed
  models_dir: d:\NLP\models
  logs_dir: d:\NLP\logs
  results_dir: d:\NLP\results
  model_name: detoxify-multilingual
  device: cpu
  language: English (Only)
  toxicity_labels: ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_attack']


In [5]:
# Verify all environment components
print("=" * 80)
print("🔍 ENVIRONMENT VERIFICATION")
print("=" * 80)

# Check PyTorch
print(f"\n✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA Available: {torch.cuda.is_available()}")
print(f"✅ Device: {CONFIG['device'].upper()}")
print(f"✅ Language Mode: {CONFIG['language']}")

# Check detoxify
try:
    model = Detoxify('multilingual')
    print(f"✅ Detoxify Model: Loaded successfully")
except Exception as e:
    print(f"❌ Detoxify: {str(e)}")

# Check directories
print(f"\n📁 Directories Created:")
for key in ['data_dir', 'models_dir', 'logs_dir', 'results_dir']:
    path = Path(CONFIG[key])
    exists = "✅" if path.exists() else "❌"
    print(f"  {exists} {key}: {CONFIG[key]}")

print("\n" + "=" * 80)
print("✅ PHASE 1 SETUP COMPLETE - Ready to proceed to Phase 2")
print("=" * 80)

🔍 ENVIRONMENT VERIFICATION

✅ PyTorch: 2.1.0+cpu
✅ CUDA Available: False
✅ Device: CPU
✅ Language Mode: English (Only)


Downloading: "https://github.com/unitaryai/detoxify/releases/download/v0.4-alpha/multilingual_debiased-0b549669.ckpt" to C:\Users\kumar/.cache\torch\hub\checkpoints\multilingual_debiased-0b549669.ckpt
100%|██████████| 1.04G/1.04G [05:16<00:00, 3.52MB/s]


❌ Detoxify: We couldn't connect to 'https://huggingface.co' to load this file, couldn't find it in the cached files and it looks like xlm-roberta-base is not the path to a directory containing a file named config.json.
Checkout your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#offline-mode'.

📁 Directories Created:
  ✅ data_dir: d:\NLP\data
  ✅ models_dir: d:\NLP\models
  ✅ logs_dir: d:\NLP\logs
  ✅ results_dir: d:\NLP\results

✅ PHASE 1 SETUP COMPLETE - Ready to proceed to Phase 2


# 🧹 PHASE 2: Text Preprocessing

In [6]:
import re
import string

class TextPreprocessor:
    """Clean and normalize text for toxicity classification (English only)."""
    
    def __init__(self, remove_urls: bool = True, remove_emails: bool = True, 
                 remove_special_chars: bool = False, lowercase: bool = True):
        """Initialize preprocessor with options."""
        self.remove_urls = remove_urls
        self.remove_emails = remove_emails
        self.remove_special_chars = remove_special_chars
        self.lowercase = lowercase
    
    def preprocess(self, text: str) -> str:
        """Apply all preprocessing steps."""
        if not text:
            return ""
        
        # Remove URLs
        if self.remove_urls:
            text = re.sub(r'http\S+|www.\S+', '', text)
        
        # Remove emails
        if self.remove_emails:
            text = re.sub(r'\S+@\S+', '', text)
        
        # Remove extra whitespace (multiple spaces to single space)
        text = re.sub(r'\s+', ' ', text)
        
        # Remove leading/trailing whitespace
        text = text.strip()
        
        # Remove special characters (keep alphanumeric, spaces, common punctuation)
        if self.remove_special_chars:
            text = re.sub(r'[^a-zA-Z0-9\s\.\,\!\?\-\']', '', text)
        
        # Convert to lowercase
        if self.lowercase:
            text = text.lower()
        
        return text
    
    def preprocess_batch(self, texts: list) -> list:
        """Preprocess a batch of texts."""
        return [self.preprocess(text) for text in texts]

# Initialize text preprocessor
text_preprocessor = TextPreprocessor(
    remove_urls=True,
    remove_emails=True,
    remove_special_chars=False,  # Keep special chars for toxicity signals
    lowercase=True
)

print("✅ Text Preprocessor initialized successfully!")

✅ Text Preprocessor initialized successfully!


In [7]:
# Test preprocessing with English sentences
test_texts = [
    "This is a great product!!!",
    "Visit us at https://example.com for more info",
    "Contact me: email@example.com",
    "I  have   multiple   spaces",
    "UPPERCASE TEXT TO LOWERCASE",
    "This is a toxic comment with bad language!@#$",
    "Normal sentence with punctuation.",
]

print("="*80)
print("📝 TEXT PREPROCESSING TEST")
print("="*80)

results = []
for text in test_texts:
    processed = text_preprocessor.preprocess(text)
    results.append({
        'original': text,
        'processed': processed
    })
    print(f"\nOriginal:\n  {text}")
    print(f"Processed:\n  {processed}")

print("\n" + "="*80)
print("✅ Preprocessing test complete!")
print("="*80)

📝 TEXT PREPROCESSING TEST

Original:
  This is a great product!!!
Processed:
  this is a great product!!!

Original:
  Visit us at https://example.com for more info
Processed:
  visit us at for more info

Original:
  Contact me: email@example.com
Processed:
  contact me:

Original:
  I  have   multiple   spaces
Processed:
  i have multiple spaces

Original:
  UPPERCASE TEXT TO LOWERCASE
Processed:
  uppercase text to lowercase

Original:
  This is a toxic comment with bad language!@#$
Processed:
  this is a toxic comment with bad

Original:
  Normal sentence with punctuation.
Processed:
  normal sentence with punctuation.

✅ Preprocessing test complete!


# 🤖 PHASE 3: Load Detoxify Model

In [8]:
# Initialize detoxify model
print("=" * 80)
print("🏗️  LOADING PRE-TRAINED DETOXIFY MODEL")
print("=" * 80)

print(f"\nModel: {CONFIG['model_name']}")
print(f"Device: {CONFIG['device']}")

try:
    # Load the multilingual detoxify model
    detoxify_model = Detoxify('multilingual')
    print(f"\n✅ Detoxify model loaded successfully!")
    print(f"  Model type: Multilingual transformer-based")
    print(f"  Toxicity labels: {CONFIG['toxicity_labels']}")
    print(f"  Mode: English-only processing")
except Exception as e:
    print(f"❌ Error loading model: {str(e)}")
    detoxify_model = None

print("\n" + "=" * 80)

🏗️  LOADING PRE-TRAINED DETOXIFY MODEL

Model: detoxify-multilingual
Device: cpu



✅ Detoxify model loaded successfully!
  Model type: Multilingual transformer-based
  Toxicity labels: ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_attack']
  Mode: English-only processing



In [9]:
# Test detoxify model
test_samples = {
    'Positive': "This is a great product! I really love it.",
    'Toxic': "You are so stupid and worthless!",
    'Offensive': "I hate all people from that country.",
    'Neutral': "The weather is nice today.",
    'Negative': "This movie sucks! Total waste of time.",
}

print("=" * 80)
print("🔍 DETOXIFY MODEL TEST")
print("=" * 80)

if detoxify_model is not None:
    test_results = {}
    for category, text in test_samples.items():
        # Preprocess
        processed = text_preprocessor.preprocess(text)
        
        # Predict
        predictions = detoxify_model.predict(processed)
        test_results[category] = predictions
        
        print(f"\n📝 {category}:")
        print(f"   Text: {text}")
        print(f"   Predictions:")
        for label, score in predictions.items():
            print(f"     - {label}: {score:.4f}")

else:
    print("❌ Model not loaded. Cannot perform predictions.")

print("\n" + "=" * 80)
print("✅ Model test complete!")
print("=" * 80)

🔍 DETOXIFY MODEL TEST

📝 Positive:
   Text: This is a great product! I really love it.
   Predictions:
     - toxicity: 0.0005
     - severe_toxicity: 0.0000
     - obscene: 0.0003
     - identity_attack: 0.0001
     - insult: 0.0002
     - threat: 0.0000
     - sexual_explicit: 0.0000

📝 Toxic:
   Text: You are so stupid and worthless!
   Predictions:
     - toxicity: 0.9978
     - severe_toxicity: 0.0014
     - obscene: 0.0442
     - identity_attack: 0.0036
     - insult: 0.9868
     - threat: 0.0017
     - sexual_explicit: 0.0029

📝 Offensive:
   Text: I hate all people from that country.
   Predictions:
     - toxicity: 0.8991
     - severe_toxicity: 0.0006
     - obscene: 0.0022
     - identity_attack: 0.7330
     - insult: 0.1646
     - threat: 0.0228
     - sexual_explicit: 0.0011

📝 Neutral:
   Text: The weather is nice today.
   Predictions:
     - toxicity: 0.0005
     - severe_toxicity: 0.0000
     - obscene: 0.0002
     - identity_attack: 0.0001
     - insult: 0.0002
     -

# 🔍 PHASE 4: Toxicity Detection & Predictions

In [10]:
def predict_toxicity(text, model, threshold=0.5):
    """
    Predict toxicity for a single text using detoxify.
    
    Args:
        text: Input text string (English only)
        model: Detoxify model instance
        threshold: Probability threshold (default 0.5)
    
    Returns:
        Dictionary with toxicity predictions and scores
    """
    # Preprocess text
    processed_text = text_preprocessor.preprocess(text)
    
    # Get predictions from detoxify
    predictions = model.predict(processed_text)
    
    # Extract scores and binary predictions
    scores = predictions
    predicted_labels = {label: int(score > threshold) for label, score in scores.items()}
    
    result = {
        'text': text,
        'processed_text': processed_text,
        'predictions': {},
        'detected_toxicity_types': [],
        'max_score': float(max(scores.values())),
        'is_toxic': int(max(scores.values()) > threshold)
    }
    
    # Add individual label predictions
    for label in CONFIG['toxicity_labels']:
        if label in scores:
            score = scores[label]
            predicted = predicted_labels.get(label, 0)
            result['predictions'][label] = {
                'probability': float(score),
                'predicted': int(predicted)
            }
            if predicted == 1:
                result['detected_toxicity_types'].append(label)
    
    return result

print("✅ Prediction function created successfully!")

✅ Prediction function created successfully!


In [11]:
# End-to-end testing
test_inputs = [
    "This is a great product! I really love it.",
    "You are so stupid and worthless!",
    "I hate all people from that country.",
    "The weather is nice today.",
    "This movie sucks! Total waste of time.",
]

print("=" * 80)
print("🔍 END-TO-END SYSTEM TESTING")
print("=" * 80)

if detoxify_model is not None:
    print("\n✅ Model loaded for testing")
    print(f"Device: {CONFIG['device']}")
    
    # Run predictions
    test_results_e2e = []
    for text in test_inputs:
        result = predict_toxicity(text, detoxify_model)
        test_results_e2e.append(result)
        print(f"\n📝 Input: {text}")
        print(f"   Processed: {result['processed_text']}")
        print(f"   Is Toxic: {'YES ⚠️' if result['is_toxic'] else 'NO ✅'}")
        print(f"   Max Score: {result['max_score']:.4f}")
        if result['detected_toxicity_types']:
            print(f"   Detected Types: {', '.join(result['detected_toxicity_types'])}")
        print(f"   All Scores:")
        for label, scores_dict in result['predictions'].items():
            print(f"     - {label}: {scores_dict['probability']:.4f}")
else:
    print("❌ Model not loaded. Cannot run predictions.")

print("\n" + "=" * 80)

🔍 END-TO-END SYSTEM TESTING

✅ Model loaded for testing
Device: cpu

📝 Input: This is a great product! I really love it.
   Processed: this is a great product! i really love it.
   Is Toxic: NO ✅
   Max Score: 0.0005
   All Scores:
     - obscene: 0.0003
     - threat: 0.0000
     - insult: 0.0002
     - identity_attack: 0.0001

📝 Input: You are so stupid and worthless!
   Processed: you are so stupid and worthless!
   Is Toxic: YES ⚠️
   Max Score: 0.9978
   Detected Types: insult
   All Scores:
     - obscene: 0.0442
     - threat: 0.0017
     - insult: 0.9868
     - identity_attack: 0.0036

📝 Input: I hate all people from that country.
   Processed: i hate all people from that country.
   Is Toxic: YES ⚠️
   Max Score: 0.8991
   Detected Types: identity_attack
   All Scores:
     - obscene: 0.0022
     - threat: 0.0228
     - insult: 0.1646
     - identity_attack: 0.7330

📝 Input: The weather is nice today.
   Processed: the weather is nice today.
   Is Toxic: NO ✅
   Max Score: 0.0

In [12]:
# Test model confidence and score distributions
print("=" * 80)
print("📊 MODEL CONFIDENCE ANALYSIS")
print("=" * 80)

confidence_tests = [
    ("This is clearly toxic", "High toxicity"),
    ("This is a normal comment", "Low toxicity"),
    ("Some mildly critical content here", "Medium toxicity"),
]

confidence_results = []
for text, expected in confidence_tests:
    result = predict_toxicity(text, detoxify_model, threshold=0.5)
    confidence_results.append(result)
    
    print(f"\n📝 {expected}:")
    print(f"   Text: {text}")
    print(f"   Max Confidence: {result['max_score']:.4f}")
    print(f"   Detected: {', '.join(result['detected_toxicity_types']) if result['detected_toxicity_types'] else 'None'}")

print("\n" + "=" * 80)

📊 MODEL CONFIDENCE ANALYSIS

📝 High toxicity:
   Text: This is clearly toxic
   Max Confidence: 0.0316
   Detected: None

📝 Low toxicity:
   Text: This is a normal comment
   Max Confidence: 0.0005
   Detected: None

📝 Medium toxicity:
   Text: Some mildly critical content here
   Max Confidence: 0.0009
   Detected: None



In [13]:
# Test inference speed and performance
import time

print("\n" + "=" * 80)
print("⚡ INFERENCE PERFORMANCE TEST")
print("=" * 80)

performance_texts = [
    "Short text",
    "This is a medium length text for toxicity detection",
    "This is a longer text that contains multiple sentences. It tests how the model handles extended inputs. The preprocessing and detection should still work efficiently."
]

for text in performance_texts:
    start_time = time.time()
    result = predict_toxicity(text, detoxify_model)
    inference_time = (time.time() - start_time) * 1000  # Convert to ms
    
    print(f"\nText length: {len(text)} chars")
    print(f"Inference time: {inference_time:.2f} ms")
    print(f"Result: {result['is_toxic']} - {result['detected_toxicity_types'] if result['detected_toxicity_types'] else 'Clean'}")

print("\n" + "=" * 80)


⚡ INFERENCE PERFORMANCE TEST

Text length: 10 chars
Inference time: 49.29 ms
Result: 0 - Clean

Text length: 51 chars
Inference time: 53.28 ms
Result: 0 - Clean

Text length: 166 chars
Inference time: 68.28 ms
Result: 0 - Clean



# ⚖️ PHASE 5: Severity & Post-Processing

In [14]:
def calculate_severity(predictions, scores):
    """
    Calculate overall toxicity severity level.
    
    Args:
        predictions: Dict of label predictions (0/1)
        scores: Dict of label scores (0-1)
    
    Returns:
        Severity level: 'NONE', 'LOW', 'MEDIUM', 'HIGH', 'CRITICAL'
    """
    # Calculate weighted severity
    # Severe_toxic and threat get higher weights
    weights = {
        'toxic': 1.0,
        'severe_toxic': 2.0,  # High weight
        'obscene': 1.0,
        'threat': 2.0,  # High weight
        'insult': 0.8,
        'identity_attack': 1.5  # Medium-high weight
    }
    
    weighted_score = 0
    total_weight = 0
    
    for label, predicted in predictions.items():
        if predicted == 1:
            score = scores.get(label, 0)
            weight = weights.get(label, 1.0)
            weighted_score += score * weight
            total_weight += weight
    
    if total_weight == 0:
        return 'NONE'
    
    avg_weighted = weighted_score / total_weight
    
    # Map to severity levels
    if avg_weighted < 0.3:
        return 'NONE'
    elif avg_weighted < 0.5:
        return 'LOW'
    elif avg_weighted < 0.7:
        return 'MEDIUM'
    elif avg_weighted < 0.85:
        return 'HIGH'
    else:
        return 'CRITICAL'

def format_output_json(prediction_result, add_timestamp=True):
    """
    Format prediction result as structured JSON output.
    
    Args:
        prediction_result: Output from predict_toxicity()
        add_timestamp: Whether to include timestamp
    
    Returns:
        JSON-formatted output dictionary
    """
    # Extract scores
    scores = {label: data['probability'] for label, data in prediction_result['predictions'].items()}
    predictions = {label: data['predicted'] for label, data in prediction_result['predictions'].items()}
    
    # Calculate severity
    severity = calculate_severity(predictions, scores)
    
    # Format output
    output = {
        'input': {
            'original_text': prediction_result['text'],
            'processed_text': prediction_result['processed_text'],
            'length': len(prediction_result['text'])
        },
        'toxicity_detection': {
            'is_toxic': bool(prediction_result['is_toxic']),
            'severity_level': severity,
            'max_score': round(prediction_result['max_score'], 4),
            'num_detected_types': len(prediction_result['detected_toxicity_types'])
        },
        'label_scores': {
            label: round(score, 4) 
            for label, score in scores.items()
        },
        'detected_types': prediction_result['detected_toxicity_types'],
        'timestamp': datetime.now().isoformat() if add_timestamp else None
    }
    
    return output

# Test post-processing
print("=" * 80)
print("🔄 POST-PROCESSING & SEVERITY CALCULATION")
print("=" * 80)

if 'test_results_e2e' in locals() and len(test_results_e2e) > 0:
    print("\n✅ Processing test results...")
    formatted_outputs = []
    
    for i, result in enumerate(test_results_e2e):
        json_output = format_output_json(result)
        formatted_outputs.append(json_output)
        
        print(f"\n{'─'*80}")
        print(f"Sample {i+1}:")
        print(json.dumps(json_output, indent=2))
    
    print("\n" + "=" * 80)
else:
    print("⏳ Test results not available yet.")
    print("ℹ️  Run Phase 4 predictions first.")

print("=" * 80)

🔄 POST-PROCESSING & SEVERITY CALCULATION

✅ Processing test results...

────────────────────────────────────────────────────────────────────────────────
Sample 1:
{
  "input": {
    "original_text": "This is a great product! I really love it.",
    "processed_text": "this is a great product! i really love it.",
    "length": 42
  },
  "toxicity_detection": {
    "is_toxic": false,
    "severity_level": "NONE",
    "max_score": 0.0005,
    "num_detected_types": 0
  },
  "label_scores": {
    "obscene": 0.0003,
    "threat": 0.0,
    "insult": 0.0002,
    "identity_attack": 0.0001
  },
  "detected_types": [],
  "timestamp": "2026-04-16T01:29:13.929381"
}

────────────────────────────────────────────────────────────────────────────────
Sample 2:
{
  "input": {
    "original_text": "You are so stupid and worthless!",
    "processed_text": "you are so stupid and worthless!",
    "length": 32
  },
  "toxicity_detection": {
    "is_toxic": true,
    "severity_level": "CRITICAL",
    "max_scor

In [15]:
# Test severity calculation with various toxicity levels
print("=" * 80)
print("⚖️ SEVERITY LEVEL VERIFICATION")
print("=" * 80)

severity_test_cases = [
    ("Great product!", "Expected: NONE"),
    ("This is bad", "Expected: LOW-MEDIUM"),
    ("Severe threat!", "Expected: HIGH-CRITICAL"),
    ("Hate filled message", "Expected: HIGH"),
]

for text, expected in severity_test_cases:
    result = format_output_json(predict_toxicity(text, detoxify_model))
    severity = result['toxicity_detection']['severity_level']
    
    print(f"\n📝 {text}")
    print(f"   Severity: {severity} | {expected}")
    print(f"   Score: {result['toxicity_detection']['max_score']:.4f}")
    print(f"   Types: {', '.join(result['detected_types']) if result['detected_types'] else 'None'}")

print("\n" + "=" * 80)

⚖️ SEVERITY LEVEL VERIFICATION

📝 Great product!
   Severity: NONE | Expected: NONE
   Score: 0.0005
   Types: None



📝 This is bad
   Severity: NONE | Expected: LOW-MEDIUM
   Score: 0.0334
   Types: None

📝 Severe threat!
   Severity: NONE | Expected: HIGH-CRITICAL
   Score: 0.0475
   Types: None

📝 Hate filled message
   Severity: NONE | Expected: HIGH
   Score: 0.0167
   Types: None



# 🎯 PHASE 6: Production Pipeline & Results Export

In [16]:
class ToxicityDetectionPipeline:
    """
    Production-ready toxicity detection pipeline.
    Complete workflow: Input → Preprocessing → Detection → JSON Output
    """
    
    def __init__(self, model):
        """
        Initialize pipeline with detoxify model.
        
        Args:
            model: Detoxify model instance
        """
        self.model = model
        self.preprocessor = TextPreprocessor()
        self.stats = {'processed': 0, 'toxic_detected': 0, 'average_score': 0}
        print(f"✅ Pipeline initialized with {CONFIG['model_name']} model")
    
    def process(self, text, threshold=0.5, return_json=True):
        """
        Process text through complete pipeline.
        
        Args:
            text: Input text to analyze (English only)
            threshold: Probability threshold (0-1)
            return_json: Whether to return formatted JSON or raw predictions
        
        Returns:
            JSON output or raw prediction dict
        """
        # Predict toxicity
        prediction = predict_toxicity(text, self.model, threshold=threshold)
        
        # Update stats
        self.stats['processed'] += 1
        if prediction['is_toxic']:
            self.stats['toxic_detected'] += 1
        
        if return_json:
            # Format as JSON
            json_output = format_output_json(prediction)
            return json_output
        else:
            return prediction
    
    def process_batch(self, texts, threshold=0.5, return_json=True):
        """
        Process multiple texts in batch.
        
        Args:
            texts: List of text strings (English only)
            threshold: Probability threshold
            return_json: Whether to return formatted JSON
        
        Returns:
            List of outputs (JSON or predictions)
        """
        results = []
        for text in tqdm(texts, desc="Processing batch"):
            result = self.process(text, threshold=threshold, return_json=return_json)
            results.append(result)
        return results
    
    def get_stats(self):
        """
        Get pipeline statistics.
        """
        toxic_rate = (self.stats['toxic_detected'] / self.stats['processed'] * 100) if self.stats['processed'] > 0 else 0
        return {
            'texts_processed': self.stats['processed'],
            'toxic_detected': self.stats['toxic_detected'],
            'toxic_rate': f"{toxic_rate:.1f}%"
        }

# Initialize pipeline
if detoxify_model is not None:
    pipeline = ToxicityDetectionPipeline(detoxify_model)
    print("✅ Production pipeline initialized!")
else:
    print("⏳ Model not loaded. Pipeline initialization skipped.")

✅ Pipeline initialized with detoxify-multilingual model
✅ Production pipeline initialized!


In [17]:
# Batch processing example
batch_texts = [
    "This product is amazing!",
    "You are terrible!",
    "Great service and support.",
    "I hate this so much!",
    "Excellent quality and delivery.",
]

print("=" * 80)
print("📦 BATCH PROCESSING TEST")
print("=" * 80)

if 'pipeline' in locals():
    print(f"\n📋 Processing {len(batch_texts)} texts...\n")
    
    batch_results = pipeline.process_batch(batch_texts, return_json=True)
    
    # Display results
    for i, result in enumerate(batch_results, 1):
        print(f"\n{i}. {result['input']['original_text']}")
        print(f"   Severity: {result['toxicity_detection']['severity_level']}")
        print(f"   Score: {result['toxicity_detection']['max_score']:.4f}")
        print(f"   Toxic: {'YES ⚠️' if result['toxicity_detection']['is_toxic'] else 'NO ✅'}")
    
    # Show statistics
    stats = pipeline.get_stats()
    print(f"\n{'─'*80}")
    print(f"Pipeline Statistics:")
    print(f"  Texts Processed: {stats['texts_processed']}")
    print(f"  Toxic Detected: {stats['toxic_detected']}")
    print(f"  Toxic Rate: {stats['toxic_rate']}")

else:
    print("❌ Pipeline not initialized.")

print("\n" + "=" * 80)

📦 BATCH PROCESSING TEST

📋 Processing 5 texts...



Processing batch: 100%|██████████| 5/5 [00:00<00:00, 15.17it/s]


1. This product is amazing!
   Severity: NONE
   Score: 0.0005
   Toxic: NO ✅

2. You are terrible!
   Severity: CRITICAL
   Score: 0.9592
   Toxic: YES ⚠️

3. Great service and support.
   Severity: NONE
   Score: 0.0004
   Toxic: NO ✅

4. I hate this so much!
   Severity: NONE
   Score: 0.3479
   Toxic: NO ✅

5. Excellent quality and delivery.
   Severity: NONE
   Score: 0.0004
   Toxic: NO ✅

────────────────────────────────────────────────────────────────────────────────
Pipeline Statistics:
  Texts Processed: 5
  Toxic Detected: 1
  Toxic Rate: 20.0%



In [18]:
def save_results(results, output_format='json'):
    """
    Save detection results to file.
    
    Args:
        results: List of prediction results
        output_format: 'json' or 'csv'
    """
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    if output_format == 'json':
        # Save as JSON
        output_file = RESULTS_DIR / f'toxicity_results_{timestamp}.json'
        with open(output_file, 'w') as f:
            json.dump(results, f, indent=2)
        print(f"✅ Results saved to JSON: {output_file}")
        
    elif output_format == 'csv':
        # Convert to DataFrame
        rows = []
        for result in results:
            row = {
                'text': result['input']['original_text'],
                'is_toxic': result['toxicity_detection']['is_toxic'],
                'severity': result['toxicity_detection']['severity_level'],
                'max_score': result['toxicity_detection']['max_score'],
                'detected_types': ','.join(result['detected_types'])
            }
            # Add individual scores
            for label, score in result['label_scores'].items():
                row[label] = score
            rows.append(row)
        
        df = pd.DataFrame(rows)
        output_file = RESULTS_DIR / f'toxicity_results_{timestamp}.csv'
        df.to_csv(output_file, index=False)
        print(f"✅ Results saved to CSV: {output_file}")
    
    return output_file

# Save results
print("=" * 80)
print("💾 SAVING RESULTS")
print("=" * 80)

if 'batch_results' in locals():
    print(f"\nSaving {len(batch_results)} results...\n")
    
    # Save as JSON
    json_file = save_results(batch_results, output_format='json')
    
    # Save as CSV
    csv_file = save_results(batch_results, output_format='csv')
    
    print(f"\n✅ All results saved successfully!")
else:
    print("⏳ No results to save yet.")

print("=" * 80)

💾 SAVING RESULTS

Saving 5 results...

✅ Results saved to JSON: d:\NLP\results\toxicity_results_20260416_012927.json
✅ Results saved to CSV: d:\NLP\results\toxicity_results_20260416_012927.csv

✅ All results saved successfully!


In [19]:
# System summary and model info
print("=" * 80)
print("📋 SYSTEM SUMMARY")
print("=" * 80)

print(f"""
✅ English Toxicity Detection System Ready

🎯 Configuration:
  - Language: English only
  - Model: Detoxify (Pre-trained, multilingual-optimized)
  - Toxicity Labels: {len(CONFIG['toxicity_labels'])}
  - Device: {CONFIG['device'].upper()}
  - No Training Required: Using pre-trained weights

📊 Toxicity Classes:
  1. toxic - General toxicity
  2. severe_toxic - Severe toxicity
  3. obscene - Obscene/vulgar language
  4. threat - Threatening statements
  5. insult - Insulting/derogatory language
  6. identity_attack - Attacks on identity

🔄 Processing Workflow:
  1. Input Text (English)
  2. Preprocessing (URLs, emails, whitespace normalization)
  3. Detoxify Model Inference
  4. Toxicity Score & Label Detection
  5. Severity Calculation (5 levels: NONE, LOW, MEDIUM, HIGH, CRITICAL)
  6. JSON/CSV Output Export

✨ Ready for:
  - Single predictions: pipeline.process(text)
  - Batch processing: pipeline.process_batch(texts)
  - Results export: save_results(results)
""")

print("=" * 80)

📋 SYSTEM SUMMARY

✅ English Toxicity Detection System Ready

🎯 Configuration:
  - Language: English only
  - Model: Detoxify (Pre-trained, multilingual-optimized)
  - Toxicity Labels: 6
  - Device: CPU
  - No Training Required: Using pre-trained weights

📊 Toxicity Classes:
  1. toxic - General toxicity
  2. severe_toxic - Severe toxicity
  3. obscene - Obscene/vulgar language
  4. threat - Threatening statements
  5. insult - Insulting/derogatory language
  6. identity_attack - Attacks on identity

🔄 Processing Workflow:
  1. Input Text (English)
  2. Preprocessing (URLs, emails, whitespace normalization)
  3. Detoxify Model Inference
  4. Toxicity Score & Label Detection
  5. Severity Calculation (5 levels: NONE, LOW, MEDIUM, HIGH, CRITICAL)
  6. JSON/CSV Output Export

✨ Ready for:
  - Single predictions: pipeline.process(text)
  - Batch processing: pipeline.process_batch(texts)
  - Results export: save_results(results)



# 🎬 FINAL TEST: Single Sentence Pipeline

In [20]:
# Final single sentence test through complete pipeline
print("=" * 80)
print("🎯 COMPLETE PIPELINE: SINGLE SENTENCE TEST")
print("=" * 80)

# Single test input
test_sentence = "You are such an idiot and I hate working with you!"

print(f"\n📝 Input Sentence:\n   {test_sentence}\n")

# Run through complete pipeline
if 'pipeline' in locals():
    final_result = pipeline.process(test_sentence, threshold=0.5, return_json=True)
    
    print("✅ Pipeline Processing Complete!\n")
    print("=" * 80)
    print("📊 FINAL JSON OUTPUT:")
    print("=" * 80 + "\n")
    print(json.dumps(final_result, indent=2))
    print("\n" + "=" * 80)
    
    # Summary
    print("\n🔑 KEY RESULTS:")
    print(f"  • Is Toxic: {final_result['toxicity_detection']['is_toxic']} (Predicted)")
    print(f"  • Severity Level: {final_result['toxicity_detection']['severity_level']}")
    print(f"  • Max Confidence Score: {final_result['toxicity_detection']['max_score']:.4f}")
    print(f"  • Detected Toxicity Types: {', '.join(final_result['detected_types']) if final_result['detected_types'] else 'None'}")
    print(f"\n  • Individual Scores:")
    for label, score in final_result['label_scores'].items():
        print(f"    - {label}: {score:.4f}")
else:
    print("❌ Pipeline not available")

🎯 COMPLETE PIPELINE: SINGLE SENTENCE TEST

📝 Input Sentence:
   You are such an idiot and I hate working with you!

✅ Pipeline Processing Complete!

📊 FINAL JSON OUTPUT:

{
  "input": {
    "original_text": "You are such an idiot and I hate working with you!",
    "processed_text": "you are such an idiot and i hate working with you!",
    "length": 50
  },
  "toxicity_detection": {
    "is_toxic": true,
    "severity_level": "CRITICAL",
    "max_score": 0.9976,
    "num_detected_types": 1
  },
  "label_scores": {
    "obscene": 0.4086,
    "threat": 0.005,
    "insult": 0.9818,
    "identity_attack": 0.0038
  },
  "detected_types": [
    "insult"
  ],
  "timestamp": "2026-04-16T01:29:49.167774"
}


🔑 KEY RESULTS:
  • Is Toxic: True (Predicted)
  • Severity Level: CRITICAL
  • Max Confidence Score: 0.9976
  • Detected Toxicity Types: insult

  • Individual Scores:
    - obscene: 0.4086
    - threat: 0.0050
    - insult: 0.9818
    - identity_attack: 0.0038


# 🔥 BONUS: Cuss Words & Profanity Detection Test

In [21]:
# Test with common social media cuss words and profanity
print("=" * 80)
print("🔥 SOCIAL MEDIA CUSS WORDS & PROFANITY TEST")
print("=" * 80)

# Common cuss words and profanity used on social media
cuss_words_test = [
    "Go to hell!",
    "What the fuck is this?",
    "You damn idiot!",
    "This is bullshit!",
    "Shut the fuck up!",
    "I don't give a damn!",
    "That's crap!",
    "Fuck off!",
    "You asshole!",
    "Son of a bitch!",
    "What the hell?",
    "This sucks balls!",
    "Dick head!",
    "Piss off!",
    "You're a bastard!",
    "Eat shit!",
    "Go fuck yourself!",
    "Goddamn it!",
    "This is horseshit!",
    "You douche bag!",
]

print(f"\n📋 Testing {len(cuss_words_test)} common social media cuss words...\n")

cuss_results = []
for i, text in enumerate(cuss_words_test, 1):
    result = format_output_json(predict_toxicity(text, detoxify_model, threshold=0.5))
    cuss_results.append(result)
    
    severity = result['toxicity_detection']['severity_level']
    max_score = result['toxicity_detection']['max_score']
    types = ', '.join(result['detected_types']) if result['detected_types'] else 'NONE'
    
    # Color coding for severity
    severity_emoji = {
        'NONE': '✅',
        'LOW': '🟡',
        'MEDIUM': '🟠',
        'HIGH': '🔴',
        'CRITICAL': '🚨'
    }
    
    emoji = severity_emoji.get(severity, '❓')
    
    print(f"{i:2d}. {emoji} {severity:8s} ({max_score:.4f}) - {text}")
    print(f"     Types: {types}")

print("\n" + "=" * 80)
print("📊 DETECTION SUMMARY")
print("=" * 80)

# Statistics
total = len(cuss_results)
toxic_count = sum(1 for r in cuss_results if r['toxicity_detection']['is_toxic'])
severity_counts = {}
for r in cuss_results:
    sev = r['toxicity_detection']['severity_level']
    severity_counts[sev] = severity_counts.get(sev, 0) + 1

print(f"\n✅ Total Tested: {total}")
print(f"⚠️  Toxic Detected: {toxic_count} ({toxic_count/total*100:.1f}%)")
print(f"\n📈 Severity Distribution:")
for severity in ['NONE', 'LOW', 'MEDIUM', 'HIGH', 'CRITICAL']:
    count = severity_counts.get(severity, 0)
    bar = '█' * count
    print(f"   {severity:10s}: {bar} ({count})")

# Find strongest detections
print(f"\n🔥 TOP DETECTIONS (Highest Confidence):")
sorted_results = sorted(cuss_results, key=lambda x: x['toxicity_detection']['max_score'], reverse=True)
for i, result in enumerate(sorted_results[:5], 1):
    print(f"   {i}. {result['toxicity_detection']['max_score']:.4f} - {result['input']['original_text']}")

print("\n" + "=" * 80)


🔥 SOCIAL MEDIA CUSS WORDS & PROFANITY TEST

📋 Testing 20 common social media cuss words...

 1. 🟠 MEDIUM   (0.9852) - Go to hell!
     Types: obscene
 2. 🔴 HIGH     (0.9774) - What the fuck is this?
     Types: obscene
 3. 🚨 CRITICAL (0.9975) - You damn idiot!
     Types: obscene, insult
 4. 🚨 CRITICAL (0.9569) - This is bullshit!
     Types: obscene
 5. 🔴 HIGH     (0.9919) - Shut the fuck up!
     Types: obscene, insult
 6. 🟠 MEDIUM   (0.7773) - I don't give a damn!
     Types: obscene
 7. 🔴 HIGH     (0.9499) - That's crap!
     Types: obscene
 8. 🚨 CRITICAL (0.9870) - Fuck off!
     Types: obscene
 9. 🚨 CRITICAL (0.9967) - You asshole!
     Types: obscene, insult
10. 🚨 CRITICAL (0.9928) - Son of a bitch!
     Types: obscene, insult
11. ✅ NONE     (0.8098) - What the hell?
     Types: NONE
12. 🔴 HIGH     (0.9886) - This sucks balls!
     Types: obscene, insult
13. 🔴 HIGH     (0.9900) - Dick head!
     Types: obscene, insult
14. ✅ NONE     (0.7435) - Piss off!
     Types: NONE
15. 🚨 CR

# 📊 Detailed Score Breakdown - All Cuss Words

In [22]:
print("=" * 100)
print("📊 DETAILED SCORE BREAKDOWN - EACH CUSS WORD & ALL TOXICITY LABELS")
print("=" * 100)

if 'cuss_results' in locals() and len(cuss_results) > 0:
    for idx, result in enumerate(cuss_results, 1):
        text = result['input']['original_text']
        severity = result['toxicity_detection']['severity_level']
        max_score = result['toxicity_detection']['max_score']
        detected_types = result['detected_types']
        
        # Severity emoji
        severity_emoji = {
            'NONE': '✅',
            'LOW': '🟡',
            'MEDIUM': '🟠',
            'HIGH': '🔴',
            'CRITICAL': '🚨'
        }
        emoji = severity_emoji.get(severity, '❓')
        
        print(f"\n{'─' * 100}")
        print(f"#{idx:2d} {emoji} {severity:8s} | Confidence: {max_score:.4f}")
        print(f"    Text: \"{text}\"")
        print(f"    Detected Types: {', '.join(detected_types) if detected_types else 'NONE'}")
        print(f"\n    📈 All Toxicity Scores:")
        
        # Get all 6 label scores
        label_scores = result['label_scores']
        all_labels = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_attack']
        
        for label in all_labels:
            score = label_scores.get(label, 0)
            
            # Create visual bar
            bar_length = int(score * 30)
            bar = '█' * bar_length + '░' * (30 - bar_length)
            
            # Color based on score threshold
            if score >= 0.7:
                indicator = '🔴'
            elif score >= 0.5:
                indicator = '🟠'
            elif score >= 0.3:
                indicator = '🟡'
            else:
                indicator = '✅'
            
            print(f"       {indicator} {label:15s}: {score:.4f} [{bar}]")

print("\n" + "=" * 100)
print("✅ DETAILED ANALYSIS COMPLETE")
print("=" * 100)


📊 DETAILED SCORE BREAKDOWN - EACH CUSS WORD & ALL TOXICITY LABELS

────────────────────────────────────────────────────────────────────────────────────────────────────
# 1 🟠 MEDIUM   | Confidence: 0.9852
    Text: "Go to hell!"
    Detected Types: obscene

    📈 All Toxicity Scores:
       ✅ toxic          : 0.0000 [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]
       ✅ severe_toxic   : 0.0000 [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]
       🟠 obscene        : 0.5236 [███████████████░░░░░░░░░░░░░░░]
       ✅ threat         : 0.0336 [█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]
       🟡 insult         : 0.4808 [██████████████░░░░░░░░░░░░░░░░]
       ✅ identity_attack: 0.0085 [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]

────────────────────────────────────────────────────────────────────────────────────────────────────
# 2 🔴 HIGH     | Confidence: 0.9774
    Text: "What the fuck is this?"
    Detected Types: obscene

    📈 All Toxicity Scores:
       ✅ toxic          : 0.0000 [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]
       ✅ severe_toxic   :